# Detectores de bordas por gradiente: Prewitt, Sobel e Scharr
**Grupo 05 | Tema 03** · notebook de demonstração (Victor).

Autores dos módulos: `src/utils.py` (Felipe), `src/operadores.py` (Fernando), `src/scharr_parametros.py` (Igor).

Rode as células em ordem. No Colab, a primeira célula clona o repositório e instala as dependências.

In [ ]:
# Prepara o ambiente. No Colab: clona o repositório e instala requirements. Localmente: sobe até a raiz.
import os, sys
REPO_URL = "https://github.com/SEU_USUARIO/grupo05-deteccao-bordas-gradiente.git"  # <- ajustar
if "google.colab" in sys.modules:
    if not os.path.exists("grupo05-deteccao-bordas-gradiente"):
        !git clone {REPO_URL}
    %cd grupo05-deteccao-bordas-gradiente
    !pip install -q -r requirements.txt
else:
    while not os.path.exists("src") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Pasta de trabalho:", os.getcwd())

## 0. Importações
Todas as funções vêm de `src/`, com imports a partir da raiz do repositório.

In [ ]:
import cv2, numpy as np
from src.utils import carregar_cinza, normalizar, mostrar_grade, mapa_direcao   # Felipe
from src.operadores import (KERNELS, convolucao_manual, aplicar_prewitt,         # Fernando
                            aplicar_sobel, magnitude, direcao)
from src.scharr_parametros import aplicar_scharr, adicionar_ruido, limiarizar    # Igor

## 1. Imagem original
Carregada em **tons de cinza, `uint8`** por `carregar_cinza` (Felipe). Tem bordas verticais, horizontais, diagonais e curvas.

In [ ]:
img = carregar_cinza("imagens/entrada/teste.png")
print(img.shape, img.dtype, img.min(), img.max())
mostrar_grade([img], ["Original"], colunas=1)

## 2. Convolução manual × OpenCV
`convolucao_manual` (Fernando) é NumPy puro: desliza o kernel 3×3 e soma os produtos. Se estiver certa, a diferença para `cv2.filter2D` é ≈ 0.

In [ ]:
gx_ref = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, KERNELS["sobel"][0])
gx_man = convolucao_manual(img.astype(np.float64), KERNELS["sobel"][0])
print("Diferença máxima:", np.abs(gx_ref - gx_man).max())

## 3. Gradientes Gx e Gy (Sobel)
**`cv2.CV_64F` é obrigatório:** em `uint8` os valores negativos (borda claro→escuro) seriam cortados em 0 e metade das bordas sumiria. Para exibir, `normalizar` leva o resultado para 0..255.

In [ ]:
gx, gy = aplicar_sobel(img)
print("dtype:", gx.dtype, "| Gx min/max:", gx.min(), gx.max())
mostrar_grade([normalizar(gx), normalizar(gy)], ["Gx (bordas verticais)", "Gy (bordas horizontais)"], colunas=2)

## 4. Magnitude e direção
$G=\sqrt{G_x^2+G_y^2}$ (intensidade da borda) e $\theta=\operatorname{arctan2}(G_y,G_x)$ (orientação). No mapa de direção, matiz = ângulo e brilho = magnitude.

In [ ]:
mag = magnitude(gx, gy)
theta = direcao(gx, gy)
print("theta em radianos:", theta.min(), "a", theta.max())
mostrar_grade([normalizar(mag), cv2.cvtColor(mapa_direcao(gx, gy), cv2.COLOR_BGR2RGB)],
              ["Magnitude", "Direção (cor = ângulo)"], colunas=2)

## 5. Prewitt × Sobel × Scharr
Mesma estrutura, pesos diferentes: Prewitt (1,1,1), Sobel (1,2,1), Scharr (3,10,3). Scharr é mais preciso em diagonais (análise de Igor).

In [ ]:
mags, titulos = [], []
for nome, f in [("Prewitt", aplicar_prewitt), ("Sobel", aplicar_sobel), ("Scharr", aplicar_scharr)]:
    gx_, gy_ = f(img)
    mags.append(normalizar(magnitude(gx_, gy_))); titulos.append(nome)
mostrar_grade(mags, titulos, colunas=3)

## 6. Tamanho do kernel (`ksize`)
Kernels maiores suavizam mais: bordas mais largas e menos sensíveis a ruído, com perda de detalhe fino.

In [ ]:
mags, titulos = [], []
for k in [1, 3, 5, 7]:
    gx_, gy_ = aplicar_sobel(img, ksize=k)
    mags.append(normalizar(magnitude(gx_, gy_))); titulos.append(f"ksize={k}")
mostrar_grade(mags, titulos, colunas=4)

## 7. Ruído e suavização
A derivada amplifica ruído. Um `GaussianBlur` antes reduz as bordas falsas, ao custo de engrossar as bordas reais. **Altere `sigma` e `blur` ao vivo.**

In [ ]:
sigma = 25      # <- altere ao vivo
blur = 5        # <- altere ao vivo (ímpar)
ruidosa = adicionar_ruido(img, sigma=sigma)
suave = cv2.GaussianBlur(ruidosa, (blur, blur), 0)
imagens = [ruidosa, normalizar(magnitude(*aplicar_sobel(ruidosa))),
           normalizar(magnitude(*aplicar_sobel(suave)))]
mostrar_grade(imagens, [f"Ruído σ={sigma}", "Sobel sem blur", f"Sobel com blur {blur}x{blur}"], colunas=3)

## 8. Limiar
Binariza a magnitude normalizada: baixo demais = tudo vira borda; alto demais = perde bordas fracas.

In [ ]:
limiar = 80     # <- altere ao vivo
mag_n = normalizar(mag)   # limiarizar não reescala: o limiar está na escala 0..255
mostrar_grade([mag_n, limiarizar(mag_n, 30), limiarizar(mag_n, limiar), limiarizar(mag_n, 150)],
              ["Magnitude", "Limiar 30", f"Limiar {limiar}", "Limiar 150"], colunas=4)

## 9. Conclusão rápida
- **Prewitt:** simples, pesos iguais, mais sensível a ruído.
- **Sobel:** suaviza com peso 2 no centro; padrão de uso.
- **Scharr:** mais preciso em diagonais.
- Sempre `float64` nos gradientes e suavizar antes quando há ruído.